# Продвинутый training pipeline




## Learning rate schedulers

### `StepLR`

Раньше `learning rate` оставался постоянным. `Scheduler` меняет его по заданному правилу в процессе обучения

Для простого SGD обновление выглядит так:

$$ w_{new} = w_{old} - lr \cdot \nabla_{w}L $$

`Learning rate` управляет масштабом изменения весов. В начале больший шаг может ускорить обучение, а позмже меньший - помочь уточнить решение. Но уменьшение `learning rate` не гарантирует улучшения и само по себе не устраняет переобучение

Разница с `early stopping`:

| Механизм | Что делает |
| :--- | :--- |
| Scheduler |  Меняет learning rate, обучение продолжается |
| Early stopping | Останавливает обучение, когда улучшения долго нет |


Начнем с `StepLR`: он умножает learning rate на `gamma` каждые `step_size` вызовов `scheduler.step()`. Если вызываем его рвз в конце эпохи, расписание задается в эпохах

```
scheduler = torch.optim.lr_scheduler.StepLR(
  optimizer,
  step_size=3,
  gamma=0.5,
)
```

При начальном `lr=0.001` получатся:

| Эпохи | Learning rate во время обучения |
| :--- | ---: |
| 1-3 | 0.001 |
| 4-6 | 0.0005 |
| 7-9 | 0.00025 |
| 10-12 | 0.000125|

Посмотрим на это на примере:

In [1]:
import torch

# учимся приближать один параметр к числу 3
parameter = torch.nn.Parameter(torch.tensor([0.0]))

optimizer = torch.optim.SGD(
    [parameter],
    lr=0.1,
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.5,
)

for epoch in range(1, 11):
  # LR, который используется в текущей эпохе
  lr_used = optimizer.param_groups[0]["lr"]

  optimizer.zero_grad()

  loss = (parameter - 3).square().mean()
  loss.backward()
  optimizer.step()

  # подготавливаем LR для следующей эпохи
  scheduler.step()

  lr_next = optimizer.param_groups[0]["lr"]

  print(
      f"Epoch {epoch:2d} | "
      f"LR used: {lr_used:.5f} | "
      f"LR next: {lr_next:.5f} | "
      f"Parameter after update: {parameter.item():.4f}"
  )

Epoch  1 | LR used: 0.10000 | LR next: 0.10000 | Parameter after update: 0.6000
Epoch  2 | LR used: 0.10000 | LR next: 0.10000 | Parameter after update: 1.0800
Epoch  3 | LR used: 0.10000 | LR next: 0.05000 | Parameter after update: 1.4640
Epoch  4 | LR used: 0.05000 | LR next: 0.05000 | Parameter after update: 1.6176
Epoch  5 | LR used: 0.05000 | LR next: 0.05000 | Parameter after update: 1.7558
Epoch  6 | LR used: 0.05000 | LR next: 0.02500 | Parameter after update: 1.8803
Epoch  7 | LR used: 0.02500 | LR next: 0.02500 | Parameter after update: 1.9362
Epoch  8 | LR used: 0.02500 | LR next: 0.02500 | Parameter after update: 1.9894
Epoch  9 | LR used: 0.02500 | LR next: 0.01250 | Parameter after update: 2.0400
Epoch 10 | LR used: 0.01250 | LR next: 0.01250 | Parameter after update: 2.0640


В этом примере одна эпоха содержит один шаг оптимизатора. В обычном обучении `optimizer.step()` выполняется для каждого батча, а выбранный нами `StepLR` обновляем один раз после всей эпохи

Важно обратить внимание на границу: **третья эпоха еще обучается со старым LR; уменьшенный LR используется с четвертой**. Поэтому отдельно печатаем `LR used` и `LR next`


In [5]:
# при step_size = 2 и gamma = 0.1

parameter = torch.nn.Parameter(torch.tensor([0.0]))

optimizer = torch.optim.SGD(
    [parameter],
    lr=0.1,
)

scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=2,
    gamma=0.1,
)

for epoch in range(1, 11):
  # LR, который используется в текущей эпохе
  lr_used = optimizer.param_groups[0]["lr"]

  optimizer.zero_grad()

  loss = (parameter - 3).square().mean()
  loss.backward()
  optimizer.step()

  # подготавливаем LR для следующей эпохи
  scheduler.step()

  lr_next = optimizer.param_groups[0]["lr"]

  print(
      f"Epoch {epoch:2d} | "
      f"LR used: {lr_used:.6f} | "
      f"LR next: {lr_next:.6f} | "
      f"Parameter after update: {parameter.item():.4f}"
  )

Epoch  1 | LR used: 0.100000 | LR next: 0.100000 | Parameter after update: 0.6000
Epoch  2 | LR used: 0.100000 | LR next: 0.010000 | Parameter after update: 1.0800
Epoch  3 | LR used: 0.010000 | LR next: 0.010000 | Parameter after update: 1.1184
Epoch  4 | LR used: 0.010000 | LR next: 0.001000 | Parameter after update: 1.1560
Epoch  5 | LR used: 0.001000 | LR next: 0.001000 | Parameter after update: 1.1597
Epoch  6 | LR used: 0.001000 | LR next: 0.000100 | Parameter after update: 1.1634
Epoch  7 | LR used: 0.000100 | LR next: 0.000100 | Parameter after update: 1.1638
Epoch  8 | LR used: 0.000100 | LR next: 0.000010 | Parameter after update: 1.1641
Epoch  9 | LR used: 0.000010 | LR next: 0.000010 | Parameter after update: 1.1642
Epoch 10 | LR used: 0.000010 | LR next: 0.000001 | Parameter after update: 1.1642


### `ReduceLROnPlateau`

`ReduceLROnPlateau` - уменьшение LR, когда выбранная метрика перестала улучшаться

Например, для validation loss:

```
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
  optimizer,
  mode='min',
  factor=0.5,
  patience=2,
  threshold=1e-4,
  threshold_mode='abs',
  min_lr=1e-6,
)
```

Здесь:
  - `mode='min'` - меньшее значение метрики лучше
  - `factor=0.5` - при срабатывании LR уменьшается вдвое
  - `patience=2` - допускается две проверки подряд без улучшения; уменьшение происходит на **третьей**
  - `threshold_mode='abs'` и `threshold=1e-4` - улучшением считается снижение больше чем на `0.0001` относительно лучшего значения
  - `min_lr` - нижняя граница LR

 Улучшение сбрасывает счетчик. Сравнение идет с **лучшим значением**, а не только с предыдущей эпохой. Вызываем scheduler после validation, передавая метрику: `scheduler.step(val_loss)`

 Стоит обратить внимание: прежнее early stopping останавливало обучение при `counter >= patience`. У этого scheduler снижение LR происходит при `counter > patience`. Одинаковое число `patience` здесь дает разные моменты срабатывания

 Попробуем отдельную демострацию. **Модель здесь не обучаем**: передаем заранее заданные значения loss, чтобы изучить поведение scheduler

In [8]:
import torch

parameter = torch.nn.Parameter(torch.tensor([0.0]))

optimizer = torch.optim.SGD(
    [parameter],
    lr=0.1,
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=2,
    threshold=0.0,
    threshold_mode="abs",
    min_lr=1e-6,
)

# вымышленные результаты validation
val_losses = [
    0.80,
    0.60,
    0.61,
    0.62,
    0.63,
    0.59,
    0.60,
    0.61,
    0.62,
    0.58,
]

for epoch, val_loss in enumerate(val_losses, start=1):
  lr_before = optimizer.param_groups[0]["lr"]

  scheduler.step(val_loss)

  lr_after = optimizer.param_groups[0]["lr"]

  status = "LR reduced" if lr_after < lr_before else ""

  print(
    f"Epoch {epoch:2d} | "
    f"Val loss: {val_loss:.2f} | "
    f"LR before: {lr_before:.5f} | "
    f"LR after: {lr_after:.5f} | "
    f"{status}"
  )

Epoch  1 | Val loss: 0.80 | LR before: 0.10000 | LR after: 0.10000 | 
Epoch  2 | Val loss: 0.60 | LR before: 0.10000 | LR after: 0.10000 | 
Epoch  3 | Val loss: 0.61 | LR before: 0.10000 | LR after: 0.10000 | 
Epoch  4 | Val loss: 0.62 | LR before: 0.10000 | LR after: 0.10000 | 
Epoch  5 | Val loss: 0.63 | LR before: 0.10000 | LR after: 0.05000 | LR reduced
Epoch  6 | Val loss: 0.59 | LR before: 0.05000 | LR after: 0.05000 | 
Epoch  7 | Val loss: 0.60 | LR before: 0.05000 | LR after: 0.05000 | 
Epoch  8 | Val loss: 0.61 | LR before: 0.05000 | LR after: 0.05000 | 
Epoch  9 | Val loss: 0.62 | LR before: 0.05000 | LR after: 0.02500 | LR reduced
Epoch 10 | Val loss: 0.58 | LR before: 0.02500 | LR after: 0.02500 | 


Для демонстрации поставили `threshold=0`: любое строгое снижение будет считаться улучшением, равное значение - нет

Для `ReduceLROnPlateau` важно различать два события:

  - **Новый лучший результат:** обновляется лучшая метрика и сбрасывается счетчик
  - **Срабатываение scheduler**: уменьшается LR и сбрасывается его счетчик; лучшая метрика сохраняется

Теперь согласуем scheduler с early stopping

Допустим, используем одну метрику и одинаковый критерий улучшения:

``` python
scheduler_patience = 2
early_stooping = 3
```

После трех эпох без улучшения scheduler уменьшит LR, а наше early stopping сразу остановит обучение. **Модель не успет обучиться с уменьшенным LR**

Поэтому для первого эксперимента возьмем:

``` python
scheduler_patience = 2
early_stooping = 6
```

После снижения LR модель получит еще три эпохи для попытки улучшения. При этом снижение LR **не сбрасывает отдельный счетчик early stopping** - в выбранной нами схеме его сбрасывает только улучшение метрик

